# Wave energy — input data

Visualizes the wave inputs collected so far, as they are: sites, generation profiles (2019), costs and density.
Nothing is derived for the model yet; open points are listed in `wave_energy/README.md`.
Figures are saved to `wave_energy/figures/`. Kernel: `pypsa-usa/.venv/bin/python`.

In [ ]:
import logging
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import yaml

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

HERE = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd() / "wave_energy"
ROOT = HERE.parent
DATA, FIGS = HERE / "data", HERE / "figures"
FIGS.mkdir(exist_ok=True)

sites = pd.read_csv(DATA / "raw" / "coordinates_wecc.csv").set_index("Name")
raw = pd.read_csv(DATA / "raw" / "generation_profiles.csv")
costs = pd.read_csv(DATA / "costs.csv")
density = pd.read_csv(DATA / "density.csv", index_col="scenario")
A = yaml.safe_load((DATA / "assumptions.yaml").read_text())

# power_ratio column of each site, as in the file; time = 2019-01-01 + hour
ratio = raw[[f"{s}_power_ratio" for s in sites.index]].set_axis(sites.index, axis=1)
ratio.index = pd.Timestamp("2019-01-01") + pd.to_timedelta(raw.hour, unit="h")
sites["mean_power_ratio"] = ratio.mean()
sites = sites.sort_values("Lat", ascending=False)  # north to south
ratio = ratio[sites.index]
in_ca = sites.Lat < 42.0  # California-Oregon border; the model covers California only

# Colors: categorical slots in fixed order (validated palette), text and grid recessive
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#a3a29c", "#e6e6e6"
TIER_COLOR = {"aggressive": "#86b6ef", "moderate": "#2a78d6", "conservative": "#104281"}  # cheap -> expensive
MONTHS = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "lines.linewidth": 2, "legend.frameon": False,
})


def save(fig, name):
    fig.savefig(FIGS / f"{name}.png", dpi=150, bbox_inches="tight")


sites.round(3)

## 1. Sites

Eleven sites from Washington to Santa Barbara. Filled = in California, the area the model covers.
The light-blue areas are the offshore regions PyPSA-USA uses for offshore wind, by coastal county.

In [ ]:
states = gpd.read_file(ROOT / "base" / "geo" / "state_boundaries.geojson")
offshore = gpd.read_file(ROOT / "base" / "spatial" / "offshore.geojson")

fig, ax = plt.subplots(figsize=(6, 8.5))
states[states.name.isin(["California", "Oregon", "Washington", "Nevada", "Idaho"])].plot(
    ax=ax, color="#f4f4f2", edgecolor=MUTED, linewidth=0.8)
offshore.plot(ax=ax, color="#cde2fb", edgecolor="white", linewidth=0.3)
out = sites[~in_ca]
ax.scatter(out.Lon, out.Lat, s=60, facecolor="white", edgecolor=INK2, linewidth=1.5, zorder=3,
           label="Oregon / Washington")
ax.scatter(sites.Lon[in_ca], sites.Lat[in_ca], s=70, color=SLOTS[0], edgecolor="white", linewidth=2, zorder=4,
           label="California")
for s, r in sites.iterrows():
    dy = {"WECC_D": 5, "PACWAVE": -6}.get(s, 0)  # these two are 20 km apart
    ax.annotate(s, (r.Lon, r.Lat), xytext=(8, dy), textcoords="offset points", va="center", fontsize=8,
                color=INK if in_ca[s] else INK2)
ax.set_xlim(-127.5, -117.5)
ax.set_ylim(32.3, 49.2)
ax.set_aspect(1 / np.cos(np.radians(41)))
ax.grid(False)
ax.set_xticks([]), ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.legend(loc="lower left", fontsize=8)
ax.set_title("Wave sites", loc="left", color=INK)
save(fig, "sites_map")

## 2. Mean output per site

Mean of the `power_ratio` column over 2019, as in the file (= power / 400 kW).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
colors = [SLOTS[0] if in_ca[s] else "#c3c2b7" for s in sites.index]
bars = ax.bar(sites.index, sites.mean_power_ratio * 100, color=colors, width=0.6, edgecolor="white", linewidth=2)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in sites.mean_power_ratio], padding=3, fontsize=8, color=INK2)
ax.set_ylabel("Mean power ratio (%)")
ax.set_ylim(0, 100)
ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", labelsize=8)
ax.set_title("Mean power ratio, north to south (blue = California)", loc="left", color=INK)
save(fig, "mean_power_ratio_by_site")

## 3. When they produce (California sites)

By month, and by hour of the day as written in the file (its timezone is to be checked).

In [ ]:
ca_sites = sites.index[in_ca]
color = dict(zip(ca_sites, SLOTS))
monthly = ratio[ca_sites].groupby(ratio.index.month).mean() * 100
daily = ratio[ca_sites].groupby(ratio.index.hour).mean() * 100

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.6, 1]}, sharey=True)
for s in ca_sites:
    a1.plot(monthly.index, monthly[s], color=color[s], marker="o", markersize=4, label=s)
    a2.plot(daily.index, daily[s], color=color[s], marker="o", markersize=4)
a1.set_xticks(range(1, 13), MONTHS)
a1.set_ylabel("Mean power ratio (%)")
a1.set_ylim(0, 100)
a1.set_title("By month", loc="left", color=INK)
a2.set_xticks(range(0, 24, 3), [f"{h}h" for h in range(0, 24, 3)])
a2.set_title("By hour in the file", loc="left", color=INK)
a1.legend(loc="lower left", fontsize=8, ncol=3)
fig.tight_layout()
save(fig, "seasonal_daily")

## 4. Compared with solar and wind in the model, by month

Monthly means of the model's own profiles (25-node network, 2019 weather): solar and onshore wind averaged over
all California sites, floating offshore wind over its four California sites. Wave = mean of the five California
sites above. Monthly only, so the comparison does not depend on the file's timezone.

In [ ]:
n = pypsa.Network(ROOT / "base" / "network_2035_hybrid25.nc")
pmax = n.get_switchable_as_dense("Generator", "p_max_pu")
pmax.index = pd.DatetimeIndex(pmax.index.get_level_values("timestep")).map(lambda t: t.replace(year=2019))
gens = n.generators[n.generators.bus.isin(n.buses.index[n.buses.reeds_state == "CA"])]
model = {c: pmax[gens.index[gens.carrier == c]].mean(axis=1) for c in ["solar", "onwind", "offwind_floating"]}

series = {
    "Wave (CA sites)": ratio[ca_sites].mean(axis=1),
    "Solar": model["solar"],
    "Onshore wind": model["onwind"],
    "Floating offshore wind": model["offwind_floating"],
}
fig, ax = plt.subplots(figsize=(9, 3.8))
for (name, s), c in zip(series.items(), SLOTS):
    m = s.groupby(s.index.month).mean() * 100
    ax.plot(m.index, m, color=c, marker="o", markersize=4, label=name)
ax.set_xticks(range(1, 13), MONTHS)
ax.set_ylabel("Mean output / capacity (%)")
ax.set_ylim(0, 100)
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.set_title("Monthly profile: wave vs the model's solar and wind", loc="left", color=INK)
save(fig, "wave_vs_solar_wind_monthly")

## 5. Duration curves (California sites)

Time steps sorted from highest to lowest output.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.linspace(0, 100, len(ratio))
for s in ca_sites:
    ax.plot(x, np.sort(ratio[s].values)[::-1] * 100, color=color[s], label=s)
ax.set_xlabel("Share of the year (%)")
ax.set_ylabel("Power ratio (%)")
ax.set_ylim(0, 102)
ax.legend(fontsize=8)
ax.set_title("Duration curves", loc="left", color=INK)
save(fig, "duration_curves")

## 6. Costs

The three tiers as collected (aggressive = cheapest).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
for tier, c in TIER_COLOR.items():
    t = costs[costs.scenario == tier]
    a1.plot(t.year, t.capex_musd_per_mw, color=c, marker="o", markersize=4, label=tier)
    a2.plot(t.year, t.opex_musd_per_mw_yr * 1e3, color=c, marker="o", markersize=4, label=tier)
a1.set_ylabel("CapEx (M$/MW)")
a1.set_ylim(0)
a1.set_title("CapEx", loc="left", color=INK)
a2.set_ylabel("OpEx (k$/MW-yr)")
a2.set_ylim(0)
a2.set_title("OpEx", loc="left", color=INK)
a1.legend(fontsize=8)
fig.tight_layout()
save(fig, "costs")
costs.pivot(index="year", columns="scenario")

## 7. Space needed

Density by lane width (`density.csv`), next to the value PyPSA-USA uses for floating offshore wind.

In [ ]:
cfg = yaml.safe_load((ROOT / "base" / "config_2035_hybrid25.yaml").read_text())
ow_density = cfg["renewable"]["offwind_floating"]["capacity_per_sqkm"]
dens = pd.concat([
    density.mw_per_km2.rename(lambda s: f"Wave, {s} ({density.lane_m[s]:.0f} m lanes)"),
    pd.Series({"Floating offshore wind (PyPSA-USA)": ow_density}),
])

fig, ax = plt.subplots(figsize=(8, 2.8))
colors = [TIER_COLOR[s] for s in density.index] + [MUTED]
bars = ax.barh(dens.index, dens.values, color=colors, height=0.6, edgecolor="white", linewidth=2)
ax.bar_label(bars, labels=[f"{v:g} MW/km²" for v in dens.values], padding=4, fontsize=8, color=INK2)
ax.invert_yaxis()
ax.set_xlim(0, 16)
ax.set_xlabel("Installed capacity per km² of ocean (MW/km²)")
ax.grid(axis="y", visible=False)
ax.set_title("Capacity density: wave vs floating offshore wind", loc="left", color=INK)
save(fig, "density")
density